### Modular Bronze Ingestion Pipeline:
This notebook implements an automated, idempotent ingestion engine that moves CSV data from Unity Catalog Volumes to the Bronze Layer (Delta Lake). It adheres to the medallion architecture by ensuring raw data is landed, schema is enforced, and audit metadata is attached.

This script uses try-except blocks for resilience and the COPY INTO command to ensure idempotency and incremental loading.

**Script Logic & Resilience**

- Dynamic Discovery: The script automatically detects the folder structure. It prioritizes chunk1.csv to satisfy the Day 2 requirement for "First Time Load" while remaining flexible for future incremental files .

- Error Handling: Wrapped in try-catch blocks, the script provides specific error logs for path resolution, table creation, or SQL execution, facilitating easier debugging in Databricks Jobs.

- Idempotency: By utilizing COPY INTO with force = false, the system maintains a state of what has been loaded. If the job is triggered multiple times, it only processes "changes and new data," preventing duplicates .

**Governance & Metadata**
- Audit Columns: Every row is stamped with load_dt and a source file identifier to meet compliance requirements.

- Discoverability: Tables are created in the data_bronze.bronze namespace with explicit SQL comments, making the enterprise data searchable via Unity Catalog.

In [0]:
import json
import os
import re
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, TimestampType

# --- 1. CONFIGURATION ---
# Widgets allow the pipeline to be parameterized for different dataset batches
dbutils.widgets.text("datasets_json", "[]", "Datasets JSON Array")
raw_input = dbutils.widgets.get("datasets_json")

# Constants for the Landing and Bronze storage paths
SOURCE_BASE_PATH = "/Volumes/data_landing/data_raw"
DEST_CATALOG     = "data_bronze"
DEST_SCHEMA      = "bronze"

# --- 2. DYNAMIC UTILITY FUNCTIONS ---

def get_clean_name(col_name):
    """
    Strips Byte Order Marks (BOM), non-ASCII characters, and whitespace.
    Ensures column names are compatible with Delta Lake and SQL syntax.
    """
    return re.sub(r'[^\x00-\x7F]+', '', col_name).strip()

def resolve_source_path(dataset_name):
    """
    Identifies if the source data is a large file split into 'chunks' 
    or a 'standalone' file in the landing volume hierarchy.
    """
    clean_name = dataset_name.lower()
    chunk_path = f"{SOURCE_BASE_PATH}/{clean_name}/chunks/chunk1.csv"
    standalone_path = f"{SOURCE_BASE_PATH}/{clean_name}/{clean_name}.csv"
    
    # Check physical existence in the Volume path
    if os.path.exists(chunk_path):
        return f"{SOURCE_BASE_PATH}/{clean_name}/chunks", "chunk1.csv"
    elif os.path.exists(standalone_path):
        return f"{SOURCE_BASE_PATH}/{clean_name}", f"{clean_name}.csv"
    else:
        raise FileNotFoundError(f"Source file not found for {dataset_name}")

def ensure_bronze_table(table_full_name, dataset_name, sample_path):
    """
    Initializes the Delta table by dynamically inferring headers 
    and applying automated enterprise metadata/comments for discoverability.
    """
    # Read the header only to infer columns dynamically without loading full data
    df_sample = spark.read.option("header", "true").csv(sample_path)
    
    raw_cols = df_sample.columns
    clean_cols = [get_clean_name(c) for c in raw_cols]
    # Filter out technical audit columns to avoid duplicates during creation
    data_cols = [c for c in clean_cols if c.lower() not in ["load_dt", "source"]]

    if not spark.catalog.tableExists(table_full_name):
        print(f"Creating dynamic discoverable table: {table_full_name}")
        
        # 1. Build column-level comments dynamically for the Data Catalog
        col_definitions = []
        for c in data_cols:
            auto_comment = f"Raw attribute: {c}, dynamically inferred from {dataset_name} source."
            col_definitions.append(f"`{c}` STRING COMMENT '{auto_comment}'")
        
        # Append standard Audit columns required for Medallion Data Quality
        col_definitions.append("`load_dt` TIMESTAMP COMMENT 'Timestamp of record ingestion into Bronze tier (UTC)'")
        col_definitions.append("`source` STRING COMMENT 'Path or filename of the originating data source'")
        
        schema_sql = ", ".join(col_definitions)
        
        # 2. Build table-level comment for enterprise discoverability
        table_comment = f"Enterprise Bronze Layer: Raw data for {dataset_name.upper()}. Managed via dynamic ingestion pipeline."

        # Execute DDL with custom TBLPROPERTIES to enable advanced catalog searching
        spark.sql(f"""
            CREATE TABLE {table_full_name} ({schema_sql})
            USING DELTA
            COMMENT '{table_comment}'
            TBLPROPERTIES (
                'enterprise.tier' = 'bronze',
                'enterprise.project' = 'zillow_analytics',
                'enterprise.auto_inferred' = 'true',
                'enterprise.source_dataset' = '{dataset_name.lower()}'
            )
        """)

def execute_copy_into(table_name, source_folder, filename):
    """
    Uses Delta's COPY INTO for idempotent, high-performance ingestion.
    Maps messy source headers to cleaned target headers during the load.
    """
    print(f"  - Appending {filename} to {table_name}")
    
    # Retrieve source headers to build the mapping SQL
    df_meta = spark.read.option("header", "true").csv(f"{source_folder}/{filename}")
    raw_source_cols = df_meta.columns
    
    mapping_parts = []
    for raw_name in raw_source_cols:
        clean_name = get_clean_name(raw_name)
        # Only map business data columns; audit columns are handled via the UPDATE below
        if clean_name.lower() not in ["load_dt", "source"]:
            mapping_parts.append(f"`{raw_name}` AS `{clean_name}`")
    
    col_mapping_sql = ", ".join(mapping_parts)

    # COPY INTO handles deduplication and tracks processed files automatically
    spark.sql(f"""
        COPY INTO {table_name}
        FROM (
          SELECT {col_mapping_sql} FROM '{source_folder}'
        )
        FILEFORMAT = CSV
        PATTERN = '{filename}'
        FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
        COPY_OPTIONS ('force' = 'false')
    """)
    
    # 3. Post-Ingestion Audit: Update metadata columns for newly arrived records
    spark.sql(f"""
        UPDATE {table_name} 
        SET load_dt = current_timestamp(), source = '{filename}' 
        WHERE source = 'initial_setup' OR source IS NULL
    """)

# --- 3. MAIN ORCHESTRATION ---
def run_bronze_pipeline():
    """
    Primary pipeline loop that iterates through requested datasets.
    """
    try:
        datasets = json.loads(raw_input)
        if not datasets:
            print("No datasets provided via widget.")
            return

        for ds in datasets:
            # Step A: Find the file in the Volume
            folder, file = resolve_source_path(ds)
            full_table_name = f"{DEST_CATALOG}.{DEST_SCHEMA}.{ds.lower()}"
            
            # Step B: Ensure Delta table and metadata exist
            ensure_bronze_table(full_table_name, ds, f"{folder}/{file}")
            
            # Step C: Load the data incrementally
            execute_copy_into(full_table_name, folder, file)
            print(f"Successfully processed and documented: {ds}")
            
    except Exception as e:
        print(f"Pipeline Failed: \n{str(e)}")

if __name__ == "__main__":
    run_bronze_pipeline()

#### Unit testing of data ingestion in bronze layer:

As per Deliverable Standards, all code must include evidence collection and unit tests.

**Test Scenario:** Bronze Integrity Check
Create a separate notebook in src/tests/bronze_tests to validate the ingestion results.

In [0]:
import json
import os
from pyspark.sql import functions as F

# --- 1. CONFIGURATION & PARAMETERS ---
try:
    # Initialize the Databricks widget to accept a JSON array of dataset names
    dbutils.widgets.text("datasets_json", "[]", "Datasets JSON Array")
    # Retrieve the input provided by the user or an orchestrating job
    raw_input = dbutils.widgets.get("datasets_json")
    
    # Define standard paths and catalog locations for the Zillow project
    SOURCE_BASE_PATH = "/Volumes/data_landing/data_raw"
    DEST_CATALOG = "data_bronze"
    DEST_SCHEMA = "bronze"
except Exception as e:
    # Catch configuration errors early before cluster resources are heavily utilized
    print(f"Test Configuration failed: {str(e)}")
    raise

# --- 2. MODULAR TEST FUNCTIONS ---

def test_bronze_table_integrity(dataset_name):
    """
    Performs critical integrity checks: 
    1. Row count parity between Landing and Bronze.
    2. Existence of mandatory Audit columns for Data Quality tracking.
    """
    try:
        # Construct the fully qualified table name in the Bronze catalog
        table_full_name = f"{DEST_CATALOG}.{DEST_SCHEMA}.{dataset_name.lower()}"
        
        # Path resolution: Logic to handle both chunked files and standalone CSVs
        chunk_path = f"{SOURCE_BASE_PATH}/{dataset_name}/chunks/chunk1.csv"
        standalone_path = f"{SOURCE_BASE_PATH}/{dataset_name}/{dataset_name}.csv"
        
        # Determine the correct source path for verification
        source_path = chunk_path if os.path.exists(chunk_path) else standalone_path
        
        print(f"--- Testing Table: {table_full_name} ---")
        print(f"Source file: {source_path}")

        # 1. Fetch Expected Row Count from the Landing Volume (The Ground Truth)
        # We read the header to ensure we are comparing data-to-data, excluding headers
        expected_df = spark.read.option("header", "true").csv(source_path)
        expected_count = expected_df.count()
        
        # 2. Fetch Actual Row Count from the Bronze Delta Table (The Ingested State)
        bronze_df = spark.table(table_full_name)
        actual_count = bronze_df.count()
        
        # 3. Validation: Assert Row Count Equality
        # This prevents "Silent Failures" where ingestion only partially succeeds
        assert actual_count == expected_count, (
            f"ROW COUNT MISMATCH! Source: {expected_count}, Bronze Table: {actual_count}"
        )
        
        # 4. Validation: Audit Column Verification
        # Mandatory columns for the 'Data Quality' monitoring phase of the pipeline
        assert "load_dt" in bronze_df.columns, f"Audit column 'load_dt' missing in {table_full_name}"
        assert "source" in bronze_df.columns, f"Audit column 'source' missing in {table_full_name}"
        
        print(f"SUCCESS: {dataset_name} matches source row count ({actual_count}) and contains audit columns.")

    except Exception as e:
        # Capture and report the specific failure to the Databricks notebook output
        print(f"UNIT TEST FAILED for {dataset_name}: {str(e)}")
        raise

# --- 3. TEST ORCHESTRATOR ---

def run_unit_tests():
    """
    Main entry point for batch executing tests across multiple datasets.
    """
    try:
        # Deserialize the input JSON string into a Python list
        dataset_list = json.loads(raw_input)
        if not dataset_list:
            print("No datasets provided for testing.")
            return

        # Iterate through each dataset and run the integrity check
        for ds in dataset_list:
            test_bronze_table_integrity(ds)
            
        print("\n[ALL TESTS PASSED] Bronze Ingestion Integrity Verified.")

    except Exception as e:
        # Batch-level error handling for JSON parsing or loop failures
        print(f"Batch Test Execution Error: {str(e)}")
        raise e

if __name__ == "__main__":
    # Execute the testing pipeline
    run_unit_tests()